In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
#os.environ["GRB_LICENSE_FILE"] = r"C:\Users\PC3\Desktop\gurobi.lic"
os.environ["GRB_LICENSE_FILE"] = r"C:\Users\tomas\Desktop\gurobi.lic"
import gurobipy as gp
from gurobipy import GRB 


# Data Builder

In [ ]:
from dataclasses import replace
import importlib
import uc_experiment_builders
importlib.reload(uc_experiment_builders)

from uc_experiment_builders import (
    load_network_uc_from_excel_fixed,   # or build_data_ieee_case14 if using IEEE
    diversify_demand_archetypes_preserve_C,
    scale_renewables,
    make_scenario_data,
)

# ============================================================
# 0) Build base_data ONCE (run this cell after defining paths)
# ============================================================
# Example: Gabriel Excel dataset
#XLSX_PATH = r"C:\Users\PC3\Desktop\Counterfactual-Explanations-for-optimization-problems\Mixed\UC-Experiments\RedEjemploRiesgoConfiabilidad.xlsx"  # <-- put your path

XLSX_PATH = r"RedEjemploRiesgoConfiabilidad.xlsx"

base_data = load_network_uc_from_excel_fixed(
    xlsx_path=XLSX_PATH,
    wind_scenario=2,
    solar_scenario="Alto",
    curt_penalty=1000.0,
    allow_shifting=False,
    carbon_price=0.0,
    slack_bus=0,
    rep_params=None,                      # <-- you already have these in notebook
    emission_rates=None,     # <-- you already have these
)

# ============================================================
# 1) Scenario knobs
# ============================================================
ARCHETYPE_SEED = 7
REN_SCALE = 2.0

# ============================================================
# 2) Build scenario data from base_data
# ============================================================
data, types = make_scenario_data(base_data, archetype_seed=ARCHETYPE_SEED, ren_scale=REN_SCALE)

print("Scenario ready:")
print("  ARCHETYPE_SEED =", ARCHETYPE_SEED)
print("  REN_SCALE      =", REN_SCALE)
print("  demand shape   =", data.demand.shape)
print("  #gens, #rens   =", len(data.gens), len(data.rens))


In [ ]:
from dataclasses import replace
import importlib
import uc_experiment_builders
importlib.reload(uc_experiment_builders)

from uc_experiment_builders import (
    load_network_uc_from_excel_fixed,
    make_scenario_data,
)

XLSX_PATH = r"RedEjemploRiesgoConfiabilidad.xlsx"
#XLSX_PATH = r"C:\Users\PC3\Desktop\Counterfactual-Explanations-for-optimization-problems\Mixed\UC-Experiments\RedEjemploRiesgoConfiabilidad.xlsx"


def build_data(
    *,
    wind_scenario=2,
    solar_scenario="Alto",
    curt_penalty=1000.0,
    allow_shifting=False,
    carbon_price=0.0,
    slack_bus=0,
    rep_params=None,
    emission_rates=None,
    archetype_seed=ARCHETYPE_SEED,
    ren_scale=REN_SCALE,
):
    base_data = load_network_uc_from_excel_fixed(
        xlsx_path=XLSX_PATH,
        wind_scenario=wind_scenario,
        solar_scenario=solar_scenario,
        curt_penalty=curt_penalty,
        allow_shifting=allow_shifting,
        carbon_price=carbon_price,
        slack_bus=slack_bus,
        rep_params=rep_params,
        emission_rates=emission_rates,
    )

    data, types = make_scenario_data(base_data, archetype_seed=archetype_seed, ren_scale=ren_scale)
    return data, types


In [ ]:
def compose_foils(*foils):
    """Return a foil that applies all non-None foils in sequence."""
    foils = [f for f in foils if f is not None]
    if not foils:
        return None

    def _all(m, var, *args, **kwargs):
        for f in foils:
            f(m, var, *args, **kwargs)
    return _all


def _get_var(var, name_candidates):
    """
    Helper to fetch a variable array from `var` using multiple possible keys.
    Example: _get_var(var, ["u", "U", "commit", "on"])
    """
    for nm in name_candidates:
        if isinstance(var, dict) and nm in var:
            return var[nm]
        if hasattr(var, nm):
            return getattr(var, nm)
    raise KeyError(f"Could not find any of {name_candidates} in var.")


def _as_list(x):
    if x is None:
        return []
    return list(x) if isinstance(x, (list, tuple, set)) else [x]

### Foils

In [ ]:
def foil_force_on(g, times):
    """
    Enforce u[g,t] = 1 for t in times.
    """
    times = _as_list(times)

    def _f(m, var, *args, **kwargs):
        u = _get_var(var, ["u", "U", "on", "commit"])
        for t in times:
            m.addConstr(u[g, t] == 1, name=f"foil_force_on_g{g}_t{t}")
    return _f

def foil_force_off(g, times):
    """
    Enforce u[g,t] = 0 for t in times.
    """
    times = _as_list(times)

    def _f(m, var, *args, **kwargs):
        u = _get_var(var, ["u", "U", "on", "commit"])
        for t in times:
            m.addConstr(u[g, t] == 0, name=f"foil_force_off_g{g}_t{t}")
    return _f


def foil_force_startup(g, t):
    """
    Enforce v[g,t] = 1 (startup event). Requires startup var v.
    """
    def _f(m, var, *args, **kwargs):
        v = _get_var(var, ["v", "V", "startup"])
        m.addConstr(v[g, t] == 1, name=f"foil_force_startup_g{g}_t{t}")
    return _f

def foil_force_shutdown(g, t):
    """
    Enforce w[g,t] = 1 (shutdown event). Requires shutdown var w.
    """
    def _f(m, var, *args, **kwargs):
        w = _get_var(var, ["w", "W", "shutdown"])
        m.addConstr(w[g, t] == 1, name=f"foil_force_shutdown_g{g}_t{t}")
    return _f



In [ ]:
def foil_flip_one_commitment(sol_factual, g, t):
    """
    If factual u[g,t]=1 -> enforce u[g,t]=0.
    If factual u[g,t]=0 -> enforce u[g,t]=1.
    """
    uF = sol_factual["u"]
    target = 1 - int(round(uF[g, t]))

    def _f(m, var, *args, **kwargs):
        u = _get_var(var, ["u", "U", "on", "commit"])
        m.addConstr(u[g, t] == target, name=f"foil_flip_u_g{g}_t{t}")
    return _f

def foil_flip_unit_window(sol_factual, g, t0, t1):
    """
    Enforce u[g,t] = 1-uF[g,t] for all t in [t0,t1].
    """
    uF = sol_factual["u"]

    def _f(m, var, *args, **kwargs):
        u = _get_var(var, ["u", "U", "on", "commit"])
        for t in range(t0, t1 + 1):
            target = 1 - int(round(uF[g, t]))
            m.addConstr(u[g, t] == target, name=f"foil_flipwin_u_g{g}_t{t}")
    return _f

def foil_change_on_hours(sol_factual, g, delta_on_hours):
    """
    Enforce sum_t u[g,t] = sum_t uF[g,t] + delta_on_hours.
    Positive delta => more committed hours, negative => fewer.
    """
    uF = sol_factual["u"]
    target = int(round(np.sum(uF[g, :])) + int(delta_on_hours))

    def _f(m, var, *args, **kwargs):
        u = _get_var(var, ["u", "U", "on", "commit"])
        T = u.shape[1]
        m.addConstr(gp.quicksum(u[g, t] for t in range(T)) == target,
                    name=f"foil_onhours_g{g}_d{delta_on_hours}")
    return _f


In [ ]:
def foil_at_least_k_on(units, t, k):
    """
    Enforce sum_{g in units} u[g,t] >= k.
    """
    units = _as_list(units)

    def _f(m, var, *args, **kwargs):
        u = _get_var(var, ["u", "U", "on", "commit"])
        m.addConstr(gp.quicksum(u[g, t] for g in units) >= int(k),
                    name=f"foil_kon_t{t}_k{k}")
    return _f

def foil_at_most_k_on(units, t, k):
    """
    Enforce sum_{g in units} u[g,t] <= k.
    """
    units = _as_list(units)

    def _f(m, var, *args, **kwargs):
        u = _get_var(var, ["u", "U", "on", "commit"])
        m.addConstr(gp.quicksum(u[g, t] for g in units) <= int(k),
                    name=f"foil_koff_t{t}_k{k}")
    return _f

def foil_min_startups(t0, t1, min_startups, gens=None):
    """
    Enforce sum_{g in gens, t in [t0,t1]} v[g,t] >= min_startups.
    """
    def _f(m, var, *args, **kwargs):
        v = _get_var(var, ["v", "V", "startup"])
        nG, T = v.shape
        G = range(nG) if gens is None else _as_list(gens)
        m.addConstr(
            gp.quicksum(v[g, t] for g in G for t in range(t0, t1 + 1)) >= int(min_startups),
            name=f"foil_min_startups_{t0}_{t1}_{min_startups}"
        )
    return _f

def foil_max_startups(t0, t1, max_startups, gens=None):
    """
    Enforce sum_{g in gens, t in [t0,t1]} v[g,t] <= max_startups.
    """
    def _f(m, var, *args, **kwargs):
        v = _get_var(var, ["v", "V", "startup"])
        nG, T = v.shape
        G = range(nG) if gens is None else _as_list(gens)
        m.addConstr(
            gp.quicksum(v[g, t] for g in G for t in range(t0, t1 + 1)) <= int(max_startups),
            name=f"foil_max_startups_{t0}_{t1}_{max_startups}"
        )
    return _f


### Example

In [ ]:
from uc_pipeline import solve_uc_with_cost,build_cost_vector_network_uc,default_initial_conditions ,build_index_map_network_uc


nG = len(data.gens)
nR = len(data.rens)
nB = int(data.nB)
nL = len(data.lines)
T  = int(data.T)

idx = build_index_map_network_uc(nG=nG, nR=nR, nB=nB, nL=nL, T=T, major="t")
u_init, p_init, on_time_init, off_time_init = default_initial_conditions(data)

window_size = 2
per_bus_neutrality = True


fuel_cost_vec     = np.array([g.fuel_cost     for g in data.gens], dtype=float)
no_load_cost_vec  = np.array([g.no_load_cost  for g in data.gens], dtype=float)
su_cost_vec       = np.array([g.SU_cost       for g in data.gens], dtype=float)
sd_cost_vec       = np.array([g.SD_cost       for g in data.gens], dtype=float)

emission_rate_vec = np.array(
    [getattr(g, "emission_rate", 0.0) for g in data.gens],
    dtype=float
)

# Renewables curtailment cost (nR × T)
if nR > 0:
    curt_cost_mat = np.vstack([r.curt_cost for r in data.rens])
else:
    curt_cost_mat = np.zeros((0, T))

c0 = build_cost_vector_network_uc(
    idx=idx,
    fuel_cost=fuel_cost_vec,
    emission_rate=emission_rate_vec,
    carbon_price=float(getattr(data, "carbon_price", 0.0)),
    no_load_cost=no_load_cost_vec,
    su_cost=su_cost_vec,
    sd_cost=sd_cost_vec,
    curt_cost=curt_cost_mat,
    pi_plus=data.pi_plus,      # demand shift up price
    pi_minus=data.pi_minus,    # demand shift down price
    voll=float(getattr(data, "voll", 20000.0)),
)


mF, solF, zF = solve_uc_with_cost(
    data, idx, c0,
    window_size, per_bus_neutrality,
    u_init, p_init, on_time_init, off_time_init,
    extra_constr_fn=None,
    output_flag=True
)


In [ ]:
# ── Use the data and solF already built in Cell 10 — do NOT rebuild ──
uF = solF["u"]
print("Factual on-hours per generator:")
for g in range(nG):
    on_hrs = [t for t in range(T) if int(round(uF[g, t])) == 1]
    print(f"  Gen {g}: ON at hours {on_hrs}  ({len(on_hrs)}/24)")


print("Generator parameters relevant to commitment feasibility:")
print(f"{'Gen':>4}  {'Tech-like':>10}  {'Pmin':>6}  {'Pmax':>6}  {'UT':>4}  {'DT':>4}  {'RU':>7}  {'RD':>7}  {'fuel':>8}  {'SU':>10}  {'SD':>8}")
print("-" * 95)
for g, gen in enumerate(data.gens):
    print(f"  {g:>2}  {'':>10}  {gen.Pmin:>6.1f}  {gen.Pmax:>6.1f}  {gen.UT:>4}  {gen.DT:>4}  "
          f"{gen.RU:>7.1f}  {gen.RD:>7.1f}  {gen.fuel_cost:>8.2f}  {gen.SU_cost:>10.1f}  {gen.SD_cost:>8.1f}")

print()
print("Factual commitment with UT/DT implications:")
uF = solF["u"]
for g, gen in enumerate(data.gens):
    on_hrs  = [t for t in range(T) if int(round(uF[g, t])) == 1]
    off_hrs = [t for t in range(T) if int(round(uF[g, t])) == 0]
    
    # find consecutive OFF blocks
    off_blocks = []
    if off_hrs:
        block_start = off_hrs[0]
        prev = off_hrs[0]
        for t in off_hrs[1:]:
            if t != prev + 1:
                off_blocks.append((block_start, prev, prev - block_start + 1))
                block_start = t
            prev = t
        off_blocks.append((block_start, prev, prev - block_start + 1))
    
    # find consecutive ON blocks
    on_blocks = []
    if on_hrs:
        block_start = on_hrs[0]
        prev = on_hrs[0]
        for t in on_hrs[1:]:
            if t != prev + 1:
                on_blocks.append((block_start, prev, prev - block_start + 1))
                block_start = t
            prev = t
        on_blocks.append((block_start, prev, prev - block_start + 1))

    print(f"\n  Gen {g}  (UT={gen.UT}, DT={gen.DT}, Pmin={gen.Pmin:.0f}, Pmax={gen.Pmax:.0f})")
    print(f"    ON blocks  (need >= {gen.UT}h each to be valid): {on_blocks}")
    print(f"    OFF blocks (need >= {gen.DT}h each to be valid): {off_blocks}")
    
    # flag which off blocks are long enough to allow a foil
    foilable = [(t0, t1, length) for t0, t1, length in off_blocks if length >= 2]
    dt_ok     = [(t0, t1, length) for t0, t1, length in off_blocks if length >= gen.DT]
    print(f"    OFF blocks >= 2h (long enough to insert foil): {foilable}")
    print(f"    OFF blocks >= DT={gen.DT}h (no DT violation):  {dt_ok}")

In [ ]:
from b3_ncxplain import run_ncxplain_uc

# ── Gen 4 at t=[2,3]: Diesel unit, UT=DT=1, obj=2,529,469 vs factual 2,527,699
# ── Clearly suboptimal under c0, no min-uptime/downtime blocking it
# ── This is the cleanest foil available in this scenario
foil1 = foil_force_on(g=0, times=[8, 9, 10])

# Also try Gen 1 at (9,10) — UT=DT=1, cleanest possible foil
foil2 = foil_force_on(g=4, times=[2,3,4])

# Sanity check before running
from uc_pipeline import solve_uc_with_cost, default_initial_conditions
u_init, p_init, on_time_init, off_time_init = default_initial_conditions(data)
mCheck, solCheck, zCheck = solve_uc_with_cost(
    data, idx, c0,
    window_size=2, per_bus_neutrality=True,
    u_init=u_init, p_init=p_init,
    on_time_init=on_time_init, off_time_init=off_time_init,
    extra_constr_fn=foil2, output_flag=0,
)
assert solCheck is not None, "Foil infeasible under c0 — cannot run NCXplain"
print(f"Foil feasible under c0 ✓  obj={float(np.dot(c0, zCheck)):.1f}  "
      f"(factual={float(np.dot(c0, zF)):.1f}  "
      f"gap={float(np.dot(c0, zCheck)) - float(np.dot(c0, zF)):.1f})")

out = run_ncxplain_uc(
    data=data,
    window_size=2,
    per_bus_neutrality=True,
    foil_fn=foil2,
    mutables={"gen_costs"},
    bounds={"fuel": (0, 500), "no_load": (0, 5000), "su": (0, 100000), "sd": (0, 100000)},
    weights={"gen_costs": 1.0},
    verbose=True,
)
print("status:", out["status"])

In [ ]:
from b3_ncxplain import run_ncxplain_uc
from uc_pipeline import solve_uc_with_cost, default_initial_conditions

u_init, p_init, on_time_init, off_time_init = default_initial_conditions(data)
factual_obj = float(np.dot(c0, zF))

print("Scanning all valid foil candidates (UT=DT=1 gens only):")
print(f"  {'g':>2}  {'times':>12}  {'gap':>12}  {'verdict'}")
print("  " + "-"*55)

candidates = []
for g in range(nG):
    if data.gens[g].UT > 1 or data.gens[g].DT > 1:
        continue  # skip rigid generators
    uF_g = solF["u"][g]
    off_hrs = [t for t in range(T) if int(round(uF_g[t])) == 0]
    # find all OFF blocks of length >= 2
    blocks = []
    if off_hrs:
        s = off_hrs[0]; p = off_hrs[0]
        for t in off_hrs[1:]:
            if t != p + 1:
                if p - s + 1 >= 2: blocks.append((s, p))
                s = t
            p = t
        if p - s + 1 >= 2: blocks.append((s, p))

    for (t0, t1) in blocks:
        for length in [2, 3, 4]:
            if t0 + length - 1 > t1: continue
            times = list(range(t0, t0 + length))
            test_foil = foil_force_on(g=g, times=times)
            mT, solT, zT = solve_uc_with_cost(
                data, idx, c0,
                window_size=2, per_bus_neutrality=True,
                u_init=u_init, p_init=p_init,
                on_time_init=on_time_init, off_time_init=off_time_init,
                extra_constr_fn=test_foil, output_flag=0,
            )
            if solT is None:
                print(f"  g={g}  t={times}  gap={'INFEASIBLE':>12}")
                continue
            gap = float(np.dot(c0, zT)) - factual_obj
            verdict = ("✓ good" if 100 < gap < 50000
                       else ("too trivial" if gap <= 100 else "too large"))
            print(f"  g={g}  t={times}  gap={gap:>12.1f}  {verdict}")
            if "good" in verdict:
                candidates.append((gap, g, times))

print()
if candidates:
    candidates.sort()
    best_gap, best_g, best_times = candidates[0]
    print(f"Best candidate: Gen {best_g} at t={best_times}  gap={best_gap:.1f}")
    print("Running NCXplain on best candidate...")

    foil_best = foil_force_on(g=best_g, times=best_times)
    out = run_ncxplain_uc(
        data=data,
        window_size=2,
        per_bus_neutrality=True,
        foil_fn=foil_best,
        mutables={"gen_costs"},
        bounds={"fuel": (0, 500), "no_load": (0, 5000),
                "su": (0, 100000), "sd": (0, 100000)},
        weights={"gen_costs": 1.0},
        verbose=True,
    )
    print("status:", out["status"])
else:
    print("No good candidates found — try adjusting the gap thresholds (100, 50000)")

In [ ]:
from plot_cost_changes import (
    plot_cost_changes,
    plot_cost_change_summary,
    plot_cost_change_heatmap,
)

# Optional: give your generators readable names
gen_labels = [f"Gen {g}" for g in range(len(data.gens))]
# or e.g. gen_labels = ["Coal A", "Gas B", "Hydro C", ...]

plot_cost_changes(out, data, gen_labels=gen_labels)
plot_cost_change_summary(out, data, gen_labels=gen_labels)
plot_cost_change_heatmap(out, data, gen_labels=gen_labels)

In [ ]:
from plot_comparison import plot_comparison_dashboard, plot_uc_heatmap

gen_labels = [f"Gen {g}" for g in range(len(data.gens))]
# or: gen_labels = ["Coal A", "Gas B", "Gas C", ...]

plot_comparison_dashboard(data, out)
plot_uc_heatmap(out, data, gen_labels=gen_labels)